# 1. AI Agent
- AI Agent는 단순히 질문에 답하는 LLM보다 한 단계 확장된 시스템
- 예) "오늘 서울 날씨를 검색해서 우산이 필요한지 알려줘"
    - LLM자체의 학습 지식만으로는 오늘의 실제 날씨를 알 수 없음
    - Agent는 웹 검색이나 날씨 API 같은 Tool을 호출하고, 그 결과를 다시 읽어 최종 답변을 만듦

### 핵심 구성
1. Model: 상황을 이해하고 다음 행동을 판단
2. Tools: 검색, DB 조회, 계산, API 호출 등의 실제 행동
3. State/Memory: 지금까지의 메시지와 중간 결과 보관
4. Control Flow: 어떤 순서로 실행하고 언제 반복/종료할지 결정

# 2. RAG
- 일반 RAG:
    - 질문 > 검색 > 관련 문서 > LLM > 답변
    - 검색 단계가 미리 정해져 있는 경우가 많음
- Agentic RAG: 
    - 질문 > LLM 판단 > 검색할까? 고민 > 검색 > 결과가 충분한지 판단 > 재검색/다른 Tool 도움 받음 > 답변
    - 검색 자체가 하나의 Tool이 되고, Agent가 검색 여부와 다음 행동을 판단

# 3. LangGraph

- Agent는 한 번의 LLM 호출로 끝나지 않을 수 있음
- 예) 사용자 > LLM > 검색 Tool > LLM > 계산 Tool > LLM > 최종 답변
- 위 흐름에 다음 요구사항이 붙으면 더 복잡해짐
    - 검색 결과가 부족하면 다시 검색
    - 특정 조건이면 사람의 승인 요청
    - 실패하면 재시도
    - 대화 상태 저장
    - 여러 노드를 병렬 실행

> - LangGraph는 이런 흐름을 그래프로 표현

### LangGraph의 핵심 개녕

1. State: 그래프가 실행되는 동안 여러 노드가 공유하는 데이터
2. Node: 실제 작업을 수행하는 함수
3. Edge: 한 Node 다음에 어떤 Node를 실행할지 연결
4. Conditional Edge: State를 보고 다음 경로를 동적으로 선택

In [1]:
from typing_extensions import TypedDict

In [2]:
class UserDict(TypedDict):
    id: int
    name: str
    email: str

user = {"id": 1, "name": "김사과", "email": "apple@apple.com"}

In [3]:
from pydantic import BaseModel

In [4]:
class UserModel(BaseModel):
    id: int
    name: str
    email: str

user = UserModel(id=1, name="김사과", email="apple@apple.com")
print(user)
print(user.model_dump())

id=1 name='김사과' email='apple@apple.com'
{'id': 1, 'name': '김사과', 'email': 'apple@apple.com'}


In [5]:
# pip install langgraph
from langgraph.graph import StateGraph, START, END

In [6]:
class HelloState(TypedDict):
    name: str
    greeting: str

def make_greeting(state: HelloState):  
    return {"greeting": f"안녕하세요, {state['name']}님!"}

In [7]:
builder = StateGraph(HelloState)

# 노드 추가는 함수로 해야함
builder.add_node("greeting", make_greeting)  # 그래프상 이름은 "greeging", 실제 함수는 make_greeting

# 엣지 추가
builder.add_edge(START, "greeting")  # 시작점부터 "greeting"까지의 edge 추가
builder.add_edge("greeting", END)    # "greeting"부터 END까지 edge 추가

hello_graph = builder.compile()

In [8]:
result = hello_graph.invoke({"name": "김사과", "greeting": ""})  # invoke(): 실행
print(result)

{'name': '김사과', 'greeting': '안녕하세요, 김사과님!'}


In [9]:
# 노드의 기본 패턴
# 1. State 읽기
# 2. 작업하기
# 3. 업데이트할 값 반환

class CounterState(TypedDict):
    count: int

def plus_one(state: CounterState):
    return {"count": state["count"] + 1}

builder = StateGraph(CounterState)
builder.add_node("plus_one", plus_one)
builder.add_edge(START, "plus_one")
builder.add_edge("plus_one", END)

counter_graph = builder.compile()
counter_graph.invoke({"count": 10})

{'count': 11}

### Reducer
- 여러 노드가 같은 State 필드를 업데이트할 때 기존 값을 덮어쓸지, 합칠지 정해야 함
- 기본 값은 대체(overwrite)

In [10]:
from typing import Annotated
from operator import add

In [11]:
# Annotated[list[str], add]
# list[str]: history는 문자열을 담는 리스트
# add: LangGraph가 history를 갱신할 때 기존 리스트와 새 리스트를 합치도록 지정
    # >> 새 노드가 ["A"]를 반환했을 때 기존 리스트를 버리는 것이 아니라 ["B"] + ["A"] > ["B", "A"]

class ListState(TypedDict):
    history: Annotated[list[str], add]

def first(state: ListState):
    return {"history": ["A"]}

def second(state: ListState):
    return {"history": ["B"]}

In [12]:
builder = StateGraph(ListState)

builder.add_node("first", first)
builder.add_node("second", second)

builder.add_edge(START, "first")
builder.add_edge("first", "second")
builder.add_edge("second", END)

list_graph = builder.compile()
list_graph.invoke({"history": []})

{'history': ['A', 'B']}

In [13]:
from langchain_core.messages import AnyMessage, HumanMessage, AIMessage
from langgraph.graph.message import add_messages

In [14]:
class MessageState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]  # LLM 메시지 누적

def simple_chatbot(state: MessageState):
    return {
        "messages": [AIMessage(content="Hello! How can I help?")]
    }

builder = StateGraph(MessageState)

builder.add_node("chatbot", simple_chatbot)

builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)

message_graph = builder.compile()

result = message_graph.invoke({
    "messages": [HumanMessage(content="Hello")]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

Hello
================================== Ai Message ==================================

Hello! How can I help?


In [15]:
# State
class PipelineState(TypedDict):
    text: str
    length: int

#----------------------------------

# Functions/Actions
# PipelineState 형태의 State를 state라는 매개변수로 받는다
def clean_text(state: PipelineState):
    return {"text": state["text"].strip()}

def to_upper(state: PipelineState):
    return {"text": state["text"].upper()}

def count_length(state: PipelineState):
    return {"length": len(state["text"])}

# -----------------------------------

# PipelineState를 State 구조로 사용하는 StateGraph 객체 생성
builder = StateGraph(PipelineState)

# 각 함수를 Node로 등록
builder.add_node("clean", clean_text)
builder.add_node("upper", to_upper)
builder.add_node("count", count_length)

# edge로 Node들 연결 & 실행 순서 연결
# START → clean → upper → count → END
builder.add_edge(START, "clean")
builder.add_edge("clean", "upper")
builder.add_edge("upper", "count")
builder.add_edge("count", END)

# 실행 가능한 그래프를 pipeline_graph에 저장
pipeline_graph = builder.compile()

# 초기 상태 전달 및 실제 실행
pipeline_graph.invoke({
    "text": "   hello agent   ",
    "length": 0
})

{'text': 'HELLO AGENT', 'length': 11}

In [16]:
from typing import Literal  # 어떤 타입을 리턴해줄지 선택 할 수 있게 하는것

In [17]:
class RouteState(TypedDict):
    score: int
    result: str

def check_score(state: RouteState):
    return {}

def pass_node(state: RouteState):
    return {"result": "통과"}

def fail_node(state: RouteState):
    return {"result": "재학습 필요"}

def route_score(state: RouteState) -> Literal["pass_node", "fail_node"]:
    if state['score'] >= 60:
        return "pass_node"
    return "fail_node"



In [18]:
builder = StateGraph(RouteState)
builder.add_node("check", check_score)
builder.add_node("pass_node", pass_node)
builder.add_node("fail_node", fail_node)

builder.add_edge(START, "check")
builder.add_conditional_edges("check", route_score)   # if문에 따라 pass_node 또는 fail_node로 간다.
builder.add_edge("pass_node", END)
builder.add_edge("fail_node", END)

route_graph = builder.compile()

In [19]:
print(route_graph.invoke({"score": 85, "result": ""}))
print(route_graph.invoke({"score": 45, "result": ""}))

{'score': 85, 'result': '통과'}
{'score': 45, 'result': '재학습 필요'}


In [20]:
class RetryState(TypedDict):
    count: int

def increase(state: RetryState):
    print("현재 count:", state["count"])
    return {"count": state["count"] + 1}

def continue_or_end(state: RetryState) -> Literal["increase", "__end__"]:
    if state["count"] < 20:
        return "increase"
    return END

builder = StateGraph(RetryState)
builder.add_node("increase", increase)
builder.add_edge(START, "increase")
builder.add_conditional_edges("increase", continue_or_end)

retry_graph = builder.compile()

retry_graph.invoke({"count": 0})

현재 count: 0
현재 count: 1
현재 count: 2
현재 count: 3
현재 count: 4
현재 count: 5
현재 count: 6
현재 count: 7
현재 count: 8
현재 count: 9
현재 count: 10
현재 count: 11
현재 count: 12
현재 count: 13
현재 count: 14
현재 count: 15
현재 count: 16
현재 count: 17
현재 count: 18
현재 count: 19


{'count': 20}

In [21]:
from langgraph.errors import GraphRecursionError

try:
    result = retry_graph.invoke(
        {"count": 0},
        config={"recursion_limit": 10}
    )
    print(result)
except GraphRecursionError:
    print("그래프 반복 횟수 제한을 초과했습니다.")

현재 count: 0
현재 count: 1
현재 count: 2
현재 count: 3
현재 count: 4
현재 count: 5
현재 count: 6
현재 count: 7
현재 count: 8
현재 count: 9
그래프 반복 횟수 제한을 초과했습니다.


### 병렬 Branch

In [22]:
class ParallelState(TypedDict):
    results: Annotated[list[str], add]  # 새로운 값을 합치는 Reducer

def start_node(state: ParallelState):
    return {"results": ["START"]}

def branch_a(state: ParallelState):
    return {"results": ["A"]}

def branch_b(state: ParallelState):
    return {"results": ["B"]}

def finish_node(state: ParallelState):
    return {"results": ["FINISH"]}

builder = StateGraph(ParallelState)
builder.add_node("start_node", start_node)
builder.add_node("branch_a", branch_a)
builder.add_node("branch_b", branch_b)
builder.add_node("finish", finish_node)

builder.add_edge(START, "start_node")
builder.add_edge("start_node", "branch_a")
builder.add_edge("start_node", "branch_b")
builder.add_edge("branch_a", "finish")
builder.add_edge("branch_b", "finish")
builder.add_edge("finish", END)

parallel_graph = builder.compile()
parallel_graph.invoke({"results": []})

{'results': ['START', 'A', 'B', 'FINISH']}

### 실행 방식

In [23]:
# .invoke(): 그래프가 끝난 후 최종 결과를 보여주는 함수
result = message_graph.invoke({
    "messages": [{"role": "user", "content": "안녕하세요"}]
})

result

{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='86a4c263-e111-4111-9ec2-ca418445e040'),
  AIMessage(content='Hello! How can I help?', additional_kwargs={}, response_metadata={}, id='aa40b81e-9571-4f4a-aa0d-4fd742745488', tool_calls=[], invalid_tool_calls=[])]}

In [24]:
# 중간 과정 확인 가능
for chunk in message_graph.stream(
    {"messages": [{"role": "user", "content": "안녕하세요"}]},
    stream_mode="updates"
):
    print(chunk)

{'chatbot': {'messages': [AIMessage(content='Hello! How can I help?', additional_kwargs={}, response_metadata={}, id='a669f2d5-2160-4d9d-8d27-622829e52b42', tool_calls=[], invalid_tool_calls=[])]}}


In [25]:
# async invoke (async: 비동기)
esult = await message_graph.ainvoke({
    "messages": [{"role": "user", "content": "비동기 호출 테스트"}]
})

result

{'messages': [HumanMessage(content='안녕하세요', additional_kwargs={}, response_metadata={}, id='86a4c263-e111-4111-9ec2-ca418445e040'),
  AIMessage(content='Hello! How can I help?', additional_kwargs={}, response_metadata={}, id='aa40b81e-9571-4f4a-aa0d-4fd742745488', tool_calls=[], invalid_tool_calls=[])]}

# 4. LLM과 Tool Calling
- Tool Calling에서 LLM은 Python 함수를 직접 실행하는 것이 아님
- LLM은 도구 호출 요청을 생성

```
{
    "name": "multiply",
    "args": {"a":3, "b":12}
}
```

In [26]:
from langchain_core.tools import tool

In [27]:
@tool
def add(a: int, b: int) -> int:
    """두 정수를 더합니다"""  # Docstring 필수 (LLM이 읽을 수 있게)
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """두 정수를 곱합니다"""  # Docstring
    return a * b

In [28]:
tools = [add, multiply]

print(add.invoke({"a":10, "b":20}))
print(multiply.invoke({"a":10, "b":20}))

30
200


### ChatOpenAI에 Tool 연결

In [ ]:
import os
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()

True

In [36]:
MODEL_NAME = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
MODEL_NAME

'gpt-5.6-luna'

In [37]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [40]:
llm = ChatOpenAI(model=MODEL_NAME, api_key=OPENAI_API_KEY, temperature=0, reasoning_effort="none")

# 위에서 만든 tool 목록을 llm에게 넘겨주고 알려주기
# llm이 필요하다고 생각할 때 이 tool을 호출해서 쓸지 판단하게 하는것
llm_with_tools = llm.bind_tools(tools)  

In [41]:
response = llm_with_tools.invoke("3 곱하기 12를 계산하고, 11 더하기 49도 계산해줘.")
print("content: ", response.content)
print("tool_calls: ", response.tool_calls)


content:  
tool_calls:  [{'name': 'multiply', 'args': {'a': 3, 'b': 12}, 'id': 'call_xiR0KSWOh4I2pRQTC4Re9qSE', 'type': 'tool_call'}, {'name': 'add', 'args': {'a': 11, 'b': 49}, 'id': 'call_ChHTfLRtzmM9uuwrxvFmBk67', 'type': 'tool_call'}]


- bind_tools()를 했다고 Tool이 자동으로 실행되는 것은 아님
- LLM이 tooll_calls를 생성한 뒤, 누군가 Tool을 실행해야 함

In [45]:
# MessagesState에는 messsage 필드와 add_messages reducer가 이미 준비되어 있음
# 대화형 Agent에서는 매우 편리  
from langgraph.graph import MessagesState

# ToolNode는 마지막 AIMessage의 tool_calls를 읽고 실제 Tool을 실행한 뒤, 결과를 ToolMessage 형태로 State에 추가
# 흐름: HumanMessage > LLM > AIMessage(tool_calls) > ToolNode > ToolMessage > LLM 
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition

In [46]:
def call_model(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

In [ ]:
tool_node = ToolNode(tools)

```
START > agent(LLM) > Tool 필요? 아니오 > END
                                  예  > tools > agent > ...
```

In [49]:
builder = StateGraph(MessagesState)

builder.add_node("agent", call_model)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges("agent", tools_condition)  # tool로 갈지 agent로 갈지 선택

builder.add_edge("tools", "agent")

tool_agent_graph = builder.compile()

In [50]:
result = tool_agent_graph.invoke({
    "messages": [
        {
            "role": "user",
            "content": "37 곱하기 21을 계산해줘."
        }
    ]
})

for message in result["messages"]:
    message.pretty_print()

================================ Human Message =================================

37 곱하기 21을 계산해줘.
================================== Ai Message ==================================
Tool Calls:
  multiply (call_UN4SOmHUpr2RLMFkT7xWGfuZ)
 Call ID: call_UN4SOmHUpr2RLMFkT7xWGfuZ
  Args:
    a: 37
    b: 21
================================= Tool Message =================================
Name: multiply

777
================================== Ai Message ==================================

37 × 21 = **777**


In [51]:
from langchain.agents import create_agent

simple_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "당신은 친절한 AI 수업용 에이전트입니다. "
        "계산이 필요하면 제공된 도구를 사용하세요."
    )
)

In [52]:
result = simple_agent.invoke({
    "messages": [
        {"role": "user", "content": "123과 456을 더해줘."}
    ]
})

result["messages"][-1].pretty_print()

================================== Ai Message ==================================

123 + 456 = **579**입니다.


# 5. 웹 검색 Tool 추가
- LLM이 학습한 지식만 사용하는 것이 아니라, 필요한 경우 현재 웹 정보를 Tool로 검색할 수 있도록 함

In [56]:
from langchain_tavily import TavilySearch

load_dotenv()
TAVILY_KEY = os.getenv("TAVILY_API_KEY")

web_search = TavilySearch(max_results=3)  # 결과 최대 3개
web_search.invoke({"query": "LangGraph 최신 기능"})

{'query': 'LangGraph 최신 기능',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.youtube.com/watch?v=Ipa6JNbFq4g',
   'title': 'LangChain/LangGraph V1 업데이트 이후 신기능 활용하여 에이전트 제작하기!',
   'content': 'LangChain과 LangGraph가 V1으로 대대적인 업데이트를 단행하며, 에이전트 개발의 패러다임이 새롭게 정립되고 있습니다. 본 세션에서는 실무에서 바로 활용 가능한 최신 기능들을 심층적으로 살펴봅니다. 먼저, LangSmith의 신기능인 Insights를 통해 에이전트의 성능을 데이터 기반으로 분석하고, Multi-turn Evaluation으로 복잡한 대화형 에이전트의 품질을 체계적으로 평가하는 방법을 다룹니다. 또한 Polly를 활용한 Tracing 디버깅으로 에이전트의 동작을 시각적으로 추적하고 문제를 빠르게 진단하는 실전 기법을 공유합니다. 다음으로, LangGraph V1의 핵심 신기능인 create_agent API를 통해 에이전트를 보다 직관적으로 구축하는 방법과, Built-in/Custom Middleware를 활용하여 인증, 로깅, 에러 처리 등 공통 관심사를 깔끔하게 분리하는 아키텍처 설계 패턴을 소개합니다. 마지막으로, 복잡한 태스크를 처리하는 [...] [43:18] 되죠. 그럼 짬뽕이 되기 때문에 이게 막 어떻게 보면은 막 노이즈처럼 보일 수도 있고 막 이게이 정보들이 막 막\n[43:25] 짬뽕처럼 섬기게 되니까 제대로 된 답변이 안 나올 수가 있습니다.\n[43:30] 그리고 이제 컨텍스트도 빠르게 이게 채워지게 되는 거죠. 그러다 보면은 이제 컨텍스트가 폭발을 하게 되고 토큰도 많이 잡 잡아먹게 됩니다.\n[43:38] 그래서 이제 컨텍스트 엔지니어링 해야 되는데이 랭그래프에서는 컨텍스트 엔지니어링을

In [57]:
search_agent = create_agent(
    model=llm,
    tools=[web_search],
    system_prompt=(
        "당신은 웹 검색을 사용할 수 있는 리서치 에이전트입니다. "
        "최신 정보가 필요하면 검색 도구를 사용하고, "
        "검색 결과를 바탕으로 간결하게 답하세요."
    )
)

result = search_agent.invoke({
    "messages": [
        {
            "role": "user", 
            "content": "LangGraph가 무엇인지 최신 내용을 검색해서 설명해줘."
        }
    ]
})
result["messages"][-1].pretty_print()

================================== Ai Message ==================================

## LangGraph란?

**LangGraph는 LangChain 생태계에서 제공하는 저수준(low-level) 에이전트 오케스트레이션 프레임워크이자 런타임**입니다. 장시간 실행되고 상태를 유지해야 하는 AI 에이전트와 복잡한 워크플로를 **그래프 구조**로 설계·실행할 수 있게 해줍니다.  
공식 문서 기준으로는 Python과 JavaScript/TypeScript 환경에서 사용할 수 있습니다.  
(검색 기준: 2026년 11월)

공식 문서: [LangGraph Overview](https://docs.langchain.com/oss/python/langgraph/overview)

---

## 핵심 개념

LangGraph에서는 애플리케이션을 다음 요소로 표현합니다.

- **State**: 에이전트가 공유하는 현재 상태  
  - 대화 기록
  - 검색 결과
  - 작업 진행 상황
  - 도구 실행 결과 등
- **Node**: 상태를 읽고 변경하는 작업 단위  
  - LLM 호출
  - 데이터베이스 검색
  - 외부 API 호출
  - 코드 실행 등
- **Edge**: 노드 간 이동 규칙  
  - 항상 다음 단계로 이동
  - 조건에 따라 분기
  - 이전 단계로 되돌아가기
  - 작업 종료

즉, 단순히 “LLM을 한 번 호출하는 체인”이 아니라 다음과 같은 흐름을 명시적으로 만들 수 있습니다.

```text
사용자 요청
   ↓
의도 분류
   ├─ 일반 질문 → 답변 생성
   ├─ 문서 질문 → 검색 → 답변 생성
   └─ 복잡한 작업 → 도구 실행 → 검증 → 재시도
```

---

## LangChain과의 차이

간단히 구분하면 다음과 같습니다.

| 구분 | LangChain | LangGraph |
|---|---|---|
| 주된 역할 | LLM 애플리케이션을 빠르게 구

# 6. Structured Output
- Agent 결과를 사람이 읽기만 한다면 문자열도 충분함
- 다른 시스템으로 값을 전달하려면 정해진 구조가 유리함

In [58]:
from pydantic import BaseModel, Field

class MovieResponse(BaseModel):
    title: str = Field(description="영화 제목")
    director: str = Field(description="감독")
    genre: str = Field(description="장르")
    release_year: int = Field(description="개봉 연도")

structured_llm = llm.with_structured_output(MovieResponse)

In [59]:
movie = structured_llm.invoke("영화 인터스텔라 정보를 구조화해서 알려줘.")
movie

MovieResponse(title='인터스텔라', director='크리스토퍼 놀란', genre='SF, 드라마, 모험', release_year=2014)

In [60]:
print(movie.title)
print(movie.director)
print(movie.genre)
print(movie.release_year)

인터스텔라
크리스토퍼 놀란
SF, 드라마, 모험
2014
